# Create Abel Prize Awards (PRIZE PATTERN)

Abel Prize laureates scraped from the awarding body's own site
(abelprize.no), per the runbook's prize-pattern source-authority rule.

**Awarding body:** Norwegian Academy of Science and Letters
("Det Norske Videnskaps-Akademi") — F8651541334

**Schema notes (prize pattern, monetary):**
- One row per (year × laureate). Five shared years (2004, 2008, 2015,
  2020, 2021) produce 2 rows each with `portion='1/2'`.
- Monetary prize, NOK currency, year-bounded amount:
  - 2003-2018: NOK 6,000,000
  - 2019-onwards: NOK 7,500,000
  Per-laureate amount = year_amount × portion.
- `lead_investigator` = the laureate (given_name + family_name populated;
  `affiliation.name` = institution from abelprize.no detail page).
- `funder_award_id` = `abel-{year}-{lowercased-surname}`.

**Prerequisites:**
- Run `scripts/local/abel_prize_to_s3.py` first to scrape abelprize.no
  and upload the parquet to S3.

**Data source:** https://abelprize.no/winners (+ per-year detail pages)
**S3 location:** `s3a://openalex-ingest/awards/abel_prize/abel_prize_laureates.parquet`


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.abel_prize_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/abel_prize/abel_prize_laureates.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.abel_prize_raw;

## Step 1.5: Inspect raw + amount/currency note

Abel Prize **is monetary** — the §6.7 amount-coverage check applies in
full (not waived like Fields Medal). Amount is computed at transform
time from a year-based CASE expression (NOK 6M before 2019, NOK 7.5M
after) multiplied by the per-laureate `portion`. Currency is the constant
'NOK' for all years.

Source columns from abelprize.no (100% coverage in the scraper):
`year`, `laureate_name`, `given_name`, `family_name`, `institution`,
`citation`, `slug`, `portion`. No runbook-style money-field discovery
scan needed — the parquet has no money column (amount is derived in
SQL from year + portion).

In [ ]:
%sql
DESCRIBE openalex.awards.abel_prize_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.abel_prize_raw LIMIT 5;

In [ ]:
%sql
-- Coverage check on key fields
SELECT
    COUNT(*) AS total_rows,
    COUNT(year) AS has_year,
    COUNT(laureate_name) AS has_name,
    COUNT(institution) AS has_institution,
    COUNT(citation) AS has_citation,
    COUNT(portion) AS has_portion,
    MIN(year) AS min_year,
    MAX(year) AS max_year,
    COUNT(DISTINCT year) AS distinct_years,
    collect_set(portion) AS portions
FROM openalex.awards.abel_prize_raw;

## Step 1.6: Fail-fast — verify the Norwegian Academy funder row exists

The transform in Step 2 does `CROSS JOIN openalex.common.funder WHERE
funder_id = 8651541334`. If that row isn't present, the CROSS JOIN
silently emits zero rows and the insert in Step 3 looks like it
succeeded. Assert presence here before transforming.

In [ ]:
%sql
-- Must return exactly 1 row with display_name containing 'Norske Videnskaps-Akademi'
-- (the Norwegian Academy of Science and Letters, OpenAlex display name).
-- If 0 rows, STOP and flag — the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 8651541334;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.abel_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 8651541334  -- Norwegian Academy of Science and Letters
),
year_amount AS (
    -- NOK amount by year boundary (per abelprize.no — increased from 6M to 7.5M in 2019)
    SELECT
        n.*,
        CASE
            WHEN n.year >= 2019 THEN 7500000.0
            WHEN n.year >= 2003 THEN 6000000.0
            ELSE NULL
        END AS year_amount_nok
    FROM openalex.awards.abel_prize_raw n
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':abel:', n.slug
    ))) % 9000000000 AS id,
    CONCAT('Abel Prize ', TRY_CAST(n.year AS STRING), ' — ', n.laureate_name) AS display_name,
    CASE
        WHEN n.declined AND n.citation IS NOT NULL THEN CONCAT('Declined the prize. ', n.citation)
        WHEN n.declined THEN 'Declined the prize.'
        ELSE n.citation
    END AS description,
    f.funder_id,
    CONCAT('abel-', n.slug) AS funder_award_id,
    -- Apportioned amount = year_amount × portion
    CASE
        WHEN n.portion = '1'   THEN n.year_amount_nok
        WHEN n.portion = '1/2' THEN n.year_amount_nok * 0.5
        WHEN n.portion = '1/3' THEN n.year_amount_nok / 3.0
        WHEN n.portion = '1/4' THEN n.year_amount_nok * 0.25
        ELSE n.year_amount_nok
    END AS amount,
    'NOK' AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    'Abel Prize' AS funder_scheme,
    'abelprize_no' AS provenance,
    -- Abel Prize ceremony is annually in May/June in Oslo; use May 1 as canonical date.
    TRY_TO_DATE(CONCAT(TRY_CAST(n.year AS STRING), '-05-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(TRY_CAST(n.year AS STRING), '-05-01'), 'yyyy-MM-dd') AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    TRY_CAST(n.year AS INT) AS end_year,
    struct(
        n.given_name AS given_name,
        n.family_name AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            n.institution AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.source_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':abel:', n.slug
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM year_amount n
CROSS JOIN funder_resolved f
WHERE n.slug IS NOT NULL AND n.year IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 51

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'abelprize_no' AND priority = 51;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    51 AS priority
FROM openalex.awards.abel_prize_awards;

## Step 6: Verification

Abel is monetary, so §6.7 amount-coverage check applies in full
(no waiver). Expected: 29 rows total, pct_amount = 100%, currency = NOK,
amount range NOK 3M-7.5M (3M = half of 6M for early shared years,
7.5M = full modern solo amount).

In [ ]:
%sql
SELECT COUNT(*) AS total_abel_award_rows FROM openalex.awards.abel_prize_awards;

In [ ]:
%sql
-- §6.2 Schema validation on the transformed table
DESCRIBE openalex.awards.abel_prize_awards;

In [ ]:
%sql
-- §6.3 Data completeness (runbook canonical query)
-- NOTE: for Abel Prize, expected pct_description ~100% — abelprize.no
-- publishes the citation paragraph for every laureate detail page.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.abel_prize_awards;

In [ ]:
%sql
-- §6.7 amount/currency fail-fast check (NOT waived — Abel is monetary)
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(amount), 0) AS avg_amt
FROM openalex.awards.abel_prize_awards;

In [ ]:
%sql
-- Sample inspection
SELECT id, SUBSTRING(display_name, 1, 60) AS title,
       start_year, amount, currency, funder_award_id,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 50) AS affiliation
FROM openalex.awards.abel_prize_awards
ORDER BY start_year DESC, family
LIMIT 12;

In [ ]:
%sql
-- Year distribution (annual since 2003; 5 shared years × 2 laureates each)
SELECT start_year, COUNT(*) AS laureates
FROM openalex.awards.abel_prize_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.abel_prize_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;